# 17 · Choosing a model: size, price and pinning
Level L2 · Part 2 · core · ~30 min · builds on 12, 14

## Goal
Run the same Scout questions on a small, a medium and a large Gemini model, table cost, latency and answer
quality, pick the cheapest model that is good enough, and pin the chosen model id in one config place.

## The idea
A vendor sells a **model family**: one kind of model in several sizes. Small models are cheap and fast;
large ones reason better but cost more and answer slower. You choose per task: the
**cost-latency-quality trade-off**. Rule: try the smallest model first, and move up only when it fails a
check you wrote down. Prices change, so they live in a **dated price table**. Models change too, so the
chosen id is **pinned** in one config place.

Analogy: hiring. You don't send a senior engineer to rename a file, or an intern to debug the database.
You write the job down, test candidates on it, and record who you hired.

## Picture
```mermaid
flowchart LR
    T["task + check"] --> S["small model"]
    S -->|passes| D["pick it"]
    S -->|fails| M["medium model"]
    M -->|passes| D
    M -->|fails| L["large model"] --> D
    D --> C["pin id in config<br/>+ dated price"]
```

## Step by step
### 1. Start the recorder
Model calls are recorded once to `../data/llm_cassettes/17_choosing-a-model.yaml` and replayed after that,
with no key (lesson 07).

In [1]:
import sys; sys.path.insert(0, "..")
import llm_replay
llm_replay.start("17_choosing-a-model")

LLM mode: replay (17_choosing-a-model.yaml)


### 2. Setup from lessons 07, 08, 12 and 14
Nothing new yet. First, Scout's notes and system instruction (lesson 08).

In [2]:
NOTES = [
    {"id": 1, "text": "Mars has two small moons, Phobos and Deimos."},
    {"id": 2, "text": "Venus spins backwards compared with most planets."},
    {"id": 3, "text": "A day on Mars lasts about 24 hours and 39 minutes."},
    {"id": 4, "text": "Jupiter is the largest planet in the solar system."},
    {"id": 5, "text": "The rover Perseverance landed on Mars in February 2021."},
    {"id": 6, "text": "Saturn's rings are mostly made of ice."},
]
NOTES_TEXT = "\n".join(f"Note {n['id']}: {n['text']}" for n in NOTES)
SYSTEM_TEXT = ("You are Scout, a research assistant. Answer only from the user's notes below and this chat. "
               "If the notes do not cover it, say so in one sentence. Keep answers to one sentence.\n\n" + NOTES_TEXT)
print(len(NOTES), "notes")

6 notes


Then the client (lesson 07), which retries 429s with backoff (lesson 14) because this lesson makes about
20 calls in a row, and lesson 12's `usage_tokens`.

In [3]:
import os
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv("../.env")
RETRY = types.HttpRetryOptions(attempts=6, initial_delay=2.0, max_delay=30.0)        # lesson 14
client = genai.Client(vertexai=True, api_key=os.environ.get("GOOGLE_CLOUD_API_KEY") or "replay-no-key-needed",
                      http_options=types.HttpOptions(retry_options=RETRY))
NO_AFC = types.AutomaticFunctionCallingConfig(disable=True)          # lesson 13

def usage_tokens(resp) -> tuple[int, int]:      # lesson 12: thinking is billed as output
    u = resp.usage_metadata
    return u.prompt_token_count or 0, (u.candidates_token_count or 0) + (u.thoughts_token_count or 0)

print("client and helpers ready")

client and helpers ready


### 3. One family, three sizes
Gemini's current family, smallest first, from RESEARCH.md (Gemini models page, checked 2026-10-06).
Google's docs: stable models "usually don't change"; preview models "will be deprecated with at least
2 weeks notice".

In [4]:
FAMILY = [  # (size, model id, release stage)
    ("small",  "gemini-3.5-flash-lite",  "stable"),
    ("medium", "gemini-3.8-flash",       "stable"),
    ("large",  "gemini-3.1-pro-preview", "preview"),
]
for size, model_id, stage in FAMILY:
    print(f"{size:<7}{model_id:<24}{stage}")

small  gemini-3.5-flash-lite   stable
medium gemini-3.8-flash        stable
large  gemini-3.1-pro-preview  preview


### 4. A dated price table
Lesson 12's table had one price per model. Real prices change on known dates: `gemini-3.8-flash` is on a
promotion until 2026-12-31 and doubles after. So each model keeps a list of `(valid from, input, output)`
rows, and the table records the day someone checked it and where.

In [5]:
from datetime import date

PRICES_CHECKED = date(2026, 10, 6)          # where it was read: RESEARCH.md section 1 (pricing page)
PRICES_SOURCE = "https://ai.google.dev/gemini-api/docs/pricing"
PRICES_PER_MTOK = {                         # USD per 1,000,000 tokens: (valid from, input, output)
    "gemini-3.5-flash-lite":  [(date(2026, 1, 1), 0.30, 2.50)],
    "gemini-3.8-flash":       [(date(2026, 1, 1), 0.75, 3.75), (date(2027, 1, 1), 1.50, 7.50)],
    "gemini-3.1-pro-preview": [(date(2026, 1, 1), 2.00, 12.00)],   # prompts up to 200k tokens
}

def price_on(model: str, day: date) -> tuple[float, float]:
    rows = [r for r in PRICES_PER_MTOK[model] if r[0] <= day]
    _, price_in, price_out = rows[-1]                   # the latest row already in force
    return price_in, price_out

print("3.8-flash today:", price_on("gemini-3.8-flash", date(2026, 10, 8)),
      "| in Jan 2027:", price_on("gemini-3.8-flash", date(2027, 1, 15)))

3.8-flash today: (0.75, 3.75) | in Jan 2027: (1.5, 7.5)


`cost_usd` now takes the day of the call. The smallest example, no model needed: one turn of 1,000 tokens
in and 300 out on each size. `TODAY` is fixed so this lesson prints the same thing every run; real code
uses `date.today()`.

In [6]:
TODAY = date(2026, 10, 8)

def cost_usd(model: str, input_tokens: int, output_tokens: int, day: date = TODAY) -> float:
    price_in, price_out = price_on(model, day)
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000

for size, model_id, _ in FAMILY:
    print(f"{size:<7}${cost_usd(model_id, 1_000, 300):.5f} per turn")

small  $0.00105 per turn
medium $0.00187 per turn
large  $0.00560 per turn


Same tokens, about 5 times the price from small to large. But the tokens are not the same in real life:
bigger models also *think* before answering, and thinking is billed as output. We need real calls.

A table with a date can also warn you when it is old. Prices move every few months, so 30 days is a
reasonable limit before someone re-checks the pricing page.

In [7]:
def check_prices_fresh(today: date, max_age_days: int = 30) -> None:
    age = (today - PRICES_CHECKED).days
    status = "OK" if age <= max_age_days else "STALE, re-check " + PRICES_SOURCE
    print(f"prices checked {age} days ago: {status}")

check_prices_fresh(TODAY)
check_prices_fresh(date(2027, 1, 15))

prices checked 2 days ago: OK
prices checked 101 days ago: STALE, re-check https://ai.google.dev/gemini-api/docs/pricing


### 5. Two test questions with a check each
You cannot choose a model without writing down what "good enough" means. Here: an easy look-up with one
clear answer (note 5) and a question that needs one multiplication from note 3 (24 h 39 min x 10 = 246.5 hours). Each has a check
that reads the answer.

In [8]:
TASKS = {
    "lookup": ("Which note is about a spacecraft, not a planet? Give its number.",
               lambda a: "note 5" in a.lower()),
    "math":   ("How many hours would pass on Earth during 10 Martian days? Show the number.",
               lambda a: "246" in a),
}

def scout_config(max_output_tokens: int = 1024) -> types.GenerateContentConfig:
    return types.GenerateContentConfig(system_instruction=SYSTEM_TEXT, max_output_tokens=max_output_tokens,
                                       automatic_function_calling=NO_AFC)

print(list(TASKS))

['lookup', 'math']


The output cap is 1,024 tokens, not lesson 12's 60. The medium and large models think first, and thinking
counts against `max_output_tokens` (Common mistakes shows what happens with 60).

### 6. Run every task on every size, three times
Models are not deterministic (lesson 04), so one call proves little. We ask each (task, model) pair
three times, time each call with `time.perf_counter()` (lesson 11), and keep answer, tokens, cost,
latency and whether the check passed.

In [9]:
import time

RUNS = 3
results = []
for task, (question, check) in TASKS.items():
    for size, model_id, _ in FAMILY:
        for run in range(RUNS):
            start = time.perf_counter()
            resp = client.models.generate_content(model=model_id, contents=question, config=scout_config())
            seconds = time.perf_counter() - start
            tokens_in, tokens_out = usage_tokens(resp)
            answer = (resp.text or "").strip()
            results.append({"task": task, "size": size, "model": model_id, "answer": answer,
                            "in": tokens_in, "out": tokens_out, "cost": cost_usd(model_id, tokens_in, tokens_out),
                            "seconds": round(seconds, 2), "ok": check(answer)})
print(len(results), "calls done")

18 calls done


Replay returns answers instantly, so latency measured in replay means nothing (lesson 11). The real
timings were saved to `../data/recorded/17.json` when the calls were recorded; this cell loads that
recorded output in replay mode, or loads everything if you skipped the model cell.

In [10]:
import json, pathlib
RECORDED = pathlib.Path("../data/recorded/17.json")
if "results" in globals() and llm_replay.mode() in ("record", "live"):
    RECORDED.write_text(json.dumps(results, indent=1))           # real timings from a real run
elif "results" in globals():                                       # replay: answers real, timings recorded
    for r, saved in zip(results, json.loads(RECORDED.read_text())):
        r["seconds"] = saved["seconds"]
else:
    results = json.loads(RECORDED.read_text())
print("using timings from", "this run" if llm_replay.mode() in ("record", "live") else "recorded output")

using timings from recorded output


### 7. The trade-off table
One row per (task, size): how many of the 3 runs passed, average cost (`out` includes thinking tokens),
average latency, and the first answer.

In [11]:
from statistics import mean

def summarise(results: list[dict]) -> list[dict]:
    rows = []
    for task in TASKS:
        for size, model_id, _ in FAMILY:
            rs = [r for r in results if r["task"] == task and r["model"] == model_id]
            rows.append({"task": task, "size": size, "model": model_id, "passed": sum(r["ok"] for r in rs),
                         "runs": len(rs), "out": mean(r["out"] for r in rs), "cost": mean(r["cost"] for r in rs),
                         "seconds": mean(r["seconds"] for r in rs), "answer": rs[0]["answer"]})
    return rows

table = summarise(results)
print(f"{'task':<7}{'size':<7}{'pass':>5}{'out':>6}{'cost $':>10}{'secs':>6}  first answer")
for t in table:
    print(f"{t['task']:<7}{t['size']:<7}{t['passed']:>3}/{t['runs']}{t['out']:>6.0f}{t['cost']:>10.6f}"
          f"{t['seconds']:>6.1f}  {t['answer'][:52]}")

task   size    pass   out    cost $  secs  first answer
lookup small    3/3    12  0.000077   0.6  Note 5 is about a spacecraft, not a planet.
lookup medium   3/3   357  0.001456  54.5  Note 5 is about a spacecraft, referring to the rover
lookup large    3/3   460  0.005832   7.9  Note 5 is about a spacecraft.
math   small    0/3    25  0.000111   0.6  The notes do not cover how many hours would pass on 
math   medium   3/3   680  0.002668   5.4  During 10 Martian days, approximately 246.5 hours (o
math   large    3/3   845  0.010458  13.8  During 10 Martian days, about 246.5 hours would pass


How to read it (recorded 2026-10-08):
- **Quality.** All three sizes passed the look-up. On math the small model said "the notes do not cover
  it" 3 times out of 3: it obeyed the instruction too literally instead of reasoning one step.
- **Cost.** Input was about 155 tokens everywhere. Output was not: medium and large spent 330 to 1,000
  tokens, mostly thinking, against the small model's 12 to 39. So medium costs about 20 times small per
  call, not the 2 times the price list suggests.
- **Latency.** Small answered in under a second. Medium's look-up took 2.5 s once and over 75 s twice for
  the same request, so one timing tells you little; measure several runs.

### 8. Pick the cheapest model that passes
The rule from the picture, as code: keep the models that passed every run and fit the latency budget,
then take the cheapest. If none qualifies, say so instead of guessing.

In [12]:
def choose(table: list[dict], task: str, max_seconds: float = 30.0) -> dict | None:
    ok = [t for t in table if t["task"] == task and t["passed"] == t["runs"] and t["seconds"] <= max_seconds]
    return min(ok, key=lambda t: t["cost"]) if ok else None

for task in TASKS:
    best = choose(table, task)
    print(f"{task:<7}-> {best['model'] if best else 'no model qualifies'}")

lookup -> gemini-3.5-flash-lite
math   -> gemini-3.8-flash


Two tasks, two answers. The look-up goes to `gemini-3.5-flash-lite`: it passed every run at a fraction of
the cost (medium also blew the 30 s budget). Math goes to `gemini-3.8-flash`, the cheapest model that
passed, at about a quarter of the large model's cost.

### 9. What the choice is worth at volume
One call costs fractions of a cent, so the difference looks small. Scale it to a modest product:
10,000 math questions a day for 30 days.

In [13]:
PER_MONTH = 10_000 * 30
for t in (t for t in table if t["task"] == "math"):
    print(f"{t['size']:<7}${t['cost'] * PER_MONTH:>9,.2f} a month   passed {t['passed']}/{t['runs']}")

small  $    33.31 a month   passed 0/3
medium $   800.40 a month   passed 3/3
large  $ 3,137.40 a month   passed 3/3


The small model would save about $770 a month over medium, but it gets every math answer wrong. Large is
right too, at about $2,300 a month more than medium. Measured quality first, then the cheapest that passes.

### 10. Pin the choice in one place
The decision becomes config. All model ids live in one dict, read once at startup, with the environment
allowed to override (`TUT_LLM_MODEL` in `.env`, as since lesson 07). The rest of the code asks for a
*role*, never a raw id.

In [14]:
MODELS = {
    # "LARGE" in the env var name means the bigger of the two roles, not the large model
    "scout.answer": os.environ.get("TUT_LLM_MODEL_LARGE", "gemini-3.8-flash"),   # math: chosen in step 8
    "scout.small":  os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite"),    # look-ups: chosen in step 8
}

def check_model_config(models: dict[str, str]) -> None:
    for role, model_id in models.items():
        if model_id.endswith("-latest"):
            raise ValueError(f"{role}: {model_id!r} is an alias that can change under you; pin a version")
        if model_id not in PRICES_PER_MTOK:
            raise ValueError(f"{role}: no price for {model_id!r}; add it to the dated price table")
        if "preview" in model_id:
            print(f"warning: {role} uses a preview model, which can be retired at short notice")

check_model_config(MODELS)
print(MODELS)

{'scout.answer': 'gemini-3.8-flash', 'scout.small': 'gemini-3.5-flash-lite'}


Why so strict about `-latest`? Google's docs say a latest alias "will get hot-swapped with every new release";
your app's behaviour and bill would change without a deploy. A pinned id changes only when you change it.

Now the code calls by role, and checks the response: `model_version` says which model really answered.

In [15]:
def ask(role: str, question: str) -> str:
    resp = client.models.generate_content(model=MODELS[role], contents=question, config=scout_config())
    served = resp.model_version or "?"
    if not served.startswith(MODELS[role]):          # log it loudly; never silently bill the wrong model
        print(f"WARNING: asked {MODELS[role]}, served {served}")
    tokens_in, tokens_out = usage_tokens(resp)
    print(f"[{role} -> served by {served}, ${cost_usd(MODELS[role], tokens_in, tokens_out):.6f}]")
    return resp.text.strip()

print(ask("scout.answer", "When did Perseverance land?"))

[scout.answer -> served by gemini-3.8-flash, $0.001385]
The rover Perseverance landed on Mars in February 2021.


Swapping a model is now a one-line config change plus re-running your checks, which is exactly what
lesson 53 does when a model is retired.

## At work
- **Small model first, escalate on failure.** Teams route easy requests to the cheap model and only hard
  ones up; batch and caching discounts cut the rest. → lessons 33, 41, 42 and 54
- **Models are retired on a schedule** (`claude-haiku-4-5-20251001`: "not sooner than 2026-10-15"). Keep
  ids in config and run your evals before swapping. → lesson 53
- **Choose on your own tasks, not leaderboards.** Step 5's two checks are a tiny eval; teams keep dozens
  per feature. → lesson 48
- **Thinking tokens dominate the bill** on reasoning models; an effort setting trades quality for cost.
  → lesson 40

## What just happened
- `FAMILY` listed one vendor's models from small to large, with their release stage.
- `PRICES_PER_MTOK` gained valid-from dates; `price_on` showed a price change ahead, and
  `check_prices_fresh` flagged the table as stale after 30 days.
- Two tasks with written checks ran 3 times on three sizes. The small model passed the look-up but no math
  run; medium and large passed all 6, spending hundreds of thinking tokens per call.
- `choose` picked the cheapest model that passed every run: flash-lite for the look-up, `gemini-3.8-flash`
  for math.
- `MODELS` pinned the ids by role; `check_model_config` rejected aliases and unpriced ids; `ask` confirmed
  `model_version` matched the pinned id.

## Common mistakes
**1. Using a `-latest` alias, or a model with no price.** The app runs, then the bill code crashes on a
model id the price table has never seen.

In [16]:
cost_usd("gemini-flash-latest", 159, 49)

KeyError: 'gemini-flash-latest'

Fix: run `check_model_config(MODELS)` at startup, so a bad id fails before the first user request with a
message that says what to do.

**2. A thinking model with a small output cap.** Lesson 12's cap of 60 tokens was fine for flash-lite. The
medium model spends most of the 60 tokens thinking, so the answer is cut off (or missing: `.text` is
`None`), and it fails the math check.

In [17]:
resp = client.models.generate_content(model="gemini-3.8-flash", contents=TASKS["math"][0],
                                      config=scout_config(max_output_tokens=60))
answer = resp.text or ""
print("finish_reason:", resp.candidates[0].finish_reason, "| thinking tokens:", resp.usage_metadata.thoughts_token_count)
print("answer:", repr(answer))
assert TASKS["math"][1](answer), "answer failed the math check"

finish_reason: FinishReason.MAX_TOKENS | thinking tokens: 53
answer: 'Based on the'


AssertionError: answer failed the math check

Fix: size the cap per model (thinking included), and read `finish_reason` before `.text` (lesson 13).

## Try it
Use `results` (step 6) and `table` (step 7); no new model calls.
1. Recompute the monthly cost of 10,000 math questions a day for each size at **January 2027** prices
   Hint: `table` was priced at `TODAY`, so re-price each row of `results` with
   `cost_usd(r['model'], r['in'], r['out'], day)`, then average per size. Does the choice for "math" change?
2. Your product needs answers in under 2 seconds. Call `choose(table, task, max_seconds=2.0)` for both tasks.
   What does it return for "math", and what would you do about it?

In [18]:
# your code here

Finally, stop the recorder (in record mode this saves the recording).

In [19]:
llm_replay.stop()

## Key terms
- **model family** — one vendor's models of one kind in several sizes (Gemini: flash-lite, flash, pro); bigger is usually smarter, slower and dearer.
- **cost-latency-quality trade-off** — you can't maximise all three; choose the cheapest model that passes a written check for the task.
- **dated price table** — prices per million tokens with the date each became valid, plus when and where they were checked.
- **model pinning** — fixing an exact model id (not a `-latest` alias) in one config place, so behaviour and cost change only when you change it.
- **model_version** — the field on a Gemini response naming the model that actually answered; compare it with the id you pinned.